# Export `possur` and `velsur`

Load a LAMMPS trajectory with ASE, optionally filter frames using temperatures from `log.lammps`, then export `possur` and `velsur` files for downstream codes.

System-specific behaviour is now expressed through explicit configuration variables instead of hard-coded notebook logic.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd
    package_root = cwd / "python"
elif (cwd / "lammps_md_preparation" / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd / "lammps_md_preparation"
    package_root = NOTEBOOK_DIR / "python"
else:
    raise RuntimeError("Run this notebook from the repository root or from lammps_md_preparation/.")

if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from ase.io import read
from lammps_md_preparation.utils import (
    random_sample_frames,
    read_lammps_temperature_log,
    recenter_first_atom_to_xy_center,
    select_frames_by_temperature,
    set_component_for_atom_indices,
    summarise_temperatures,
    write_possur,
    write_velsur,
)


In [ ]:
# Configuration
TRAJ_FILE = NOTEBOOK_DIR / "input" / "lammps" / "50K_traj_all.lammpstrj"
LOG_FILE = NOTEBOOK_DIR / "input" / "lammps" / "50K_log.lammps"
USE_LOG_FILE = True
READ_KWARGS = {"units": "metal"}

START_INDEX = 0
END_INDEX = None
USE_PERCENTILE_WINDOW = False
LOWER_PERCENTILE = 25.0
UPPER_PERCENTILE = 75.0
N_SNAPSHOTS = 0  # 0 means: keep all selected frames.
RANDOM_SEED = 1

RECENTER_XY = False
ZERO_Z_FOR_ATOM_INDICES = []
ZERO_VELOCITY_FOR_ATOM_INDICES = []
VELOCITY_SCALE = 1.0  # Unit-conversion factor applied to exported velocities, not the MD timestep.

OUTPUT_POSSUR = NOTEBOOK_DIR / "output" / "possur_velsur" / "possur_rand.dat"
OUTPUT_VELSUR = NOTEBOOK_DIR / "output" / "possur_velsur" / "velsur_rand.dat"


In [ ]:
trajectory = read(str(TRAJ_FILE), index=":", **READ_KWARGS)
print(f"Loaded {len(trajectory)} frames from {TRAJ_FILE}")

if USE_LOG_FILE:
    timesteps, temperatures = read_lammps_temperature_log(LOG_FILE)
    selection = select_frames_by_temperature(
        trajectory=trajectory,
        temperatures=temperatures,
        start_index=START_INDEX,
        end_index=END_INDEX,
        use_percentile_window=USE_PERCENTILE_WINDOW,
        lower_percentile=LOWER_PERCENTILE,
        upper_percentile=UPPER_PERCENTILE,
    )
    working_frames = selection.frames
    print(
        f"Selected {len(working_frames)} frames in temperature range "
        f"[{selection.lower_bound:.3f}, {selection.upper_bound:.3f}]"
    )
    print(f"Selected index span: {selection.selected_indices[:5]} ...")
    print("Temperature summary:")
    for key, value in summarise_temperatures(temperatures).items():
        if key == "count":
            print(f"- {key}: {int(value)}")
        else:
            print(f"- {key}: {value:.6f}")
else:
    stop = END_INDEX if END_INDEX is not None else len(trajectory)
    working_frames = list(trajectory[START_INDEX:stop])
    print(f"Selected {len(working_frames)} frames without temperature filtering")


In [ ]:
processed_frames = []
for atoms in working_frames:
    frame = atoms.copy()
    if RECENTER_XY:
        frame = recenter_first_atom_to_xy_center(frame)
    if ZERO_Z_FOR_ATOM_INDICES:
        frame = set_component_for_atom_indices(frame, ZERO_Z_FOR_ATOM_INDICES, component=2, value=0.0)
    processed_frames.append(frame)

sampled_frames = random_sample_frames(processed_frames, N_SNAPSHOTS, seed=RANDOM_SEED)
print(f"Writing {len(sampled_frames)} frames")

write_possur(OUTPUT_POSSUR, sampled_frames)
write_velsur(
    OUTPUT_VELSUR,
    sampled_frames,
    velocity_scale=VELOCITY_SCALE,
    zero_velocity_indices=ZERO_VELOCITY_FOR_ATOM_INDICES,
)

print(f"Generated {OUTPUT_POSSUR}")
print(f"Generated {OUTPUT_VELSUR}")
